# 02b · ABT

Monta a **ABT** (*Analytical Base Table*) juntando o **alvo** com as quatro tabelas da feature store (`02a_feature_store`): uma linha por usuário e safra.

- **População:** quem teve pelo menos uma transação nos 28 dias antes da `dtRef`.
- **Alvo:** `churn = 1` se o usuário **não** teve nenhuma transação nos 28 dias a partir da `dtRef`.
- **Rótulo da safra** (`baseSafra`): `treino` (antes da OOT), `oot`, `pos_oot` (entre a OOT e a pontuação) e `score` (a pontuar, sem alvo).

**Entrada:** `fs_transacoes`, `fs_produtos`, `fs_lives`, `fs_cursos` e `workspace.tmw_loyalty.transacoes` · **Saída:** `workspace.tmw_loyalty.abt_churn`

In [0]:
%run ./00_setup

## 1. Alvo

In [0]:
spark.sql(f"""
    SELECT DISTINCT IdCliente, to_date(DtCriacao) AS dtDia FROM {T_TRANSACOES}
""").createOrReplaceTempView('tb_dia')

spark.sql(f"""
    SELECT explode(sequence(DATE'{DT_INICIO}', DATE'{DT_SCORE}', interval 1 month)) AS dtRef
""").createOrReplaceTempView('tb_safras')

alvo = spark.sql("""
    WITH pop AS (
        SELECT DISTINCT s.dtRef, d.IdCliente
        FROM tb_safras s
        JOIN tb_dia d ON d.dtDia >= date_sub(s.dtRef, 28) AND d.dtDia < s.dtRef
    ),
    futuro AS (
        SELECT DISTINCT s.dtRef, d.IdCliente
        FROM tb_safras s
        JOIN tb_dia d ON d.dtDia >= s.dtRef AND d.dtDia < date_add(s.dtRef, 28)
    )
    SELECT p.dtRef, p.IdCliente,
           CASE WHEN f.IdCliente IS NULL THEN 1 ELSE 0 END AS churn
    FROM pop p
    LEFT JOIN futuro f ON p.dtRef = f.dtRef AND p.IdCliente = f.IdCliente
""").toPandas()

max_dia = spark.sql('SELECT MAX(dtDia) FROM tb_dia').first()[0]
print(f'Alvo: {len(alvo):,} linhas em {alvo.dtRef.nunique()} safras | último dia com dados: {max_dia}'.replace(',', '.'))

## 2. Junção com a feature store

In [0]:
ORDEM = ['dtRef', 'IdCliente', 'churn', 'diasDesdePrimeira', 'recencia', 'diasAtivosD7', 'diasAtivosD14', 'diasAtivosD28', 'diasAtivosD56', 'qtdTransacoesD28', 'pontosGanhosD28', 'freqVida', 'diasAtivosD28Anterior', 'qtdMadrugada', 'qtdManha', 'qtdTarde', 'qtdNoite', 'qtdDiasSemanaD56', 'pontosGastosD28', 'saldoPontosD28', 'saldoPontos', 'avgIntervalo', 'stdIntervalo', 'maxIntervalo', 'qtdRetornos', 'horasPorDiaD28', 'qtdRedes', 'deltaDiasD28', 'tendencia', 'razaoAtraso', 'flNovatoRisco', 'transacoesPorDiaD28', 'cvIntervalo', 'periodoDominante', 'percentilPontos', 'zScorePontos', 'gastouD28', 'share_chat', 'share_desconhecido', 'share_outros', 'share_ponei', 'share_presenca', 'share_streak', 'share_troca', 'produtoDominante', 'qtdProdutosDistintos', 'audienciaMedia', 'pctLivesPresente', 'livesPerdidas', 'fezStreak', 'diasUltimoStreak', 'qtdCursosIniciados', 'qtdCursosFinalizados', 'pctMedioCompleto', 'qtdEpsD28', 'diasDesdeUltimoEp', 'flFezCurso', 'baseSafra', 'flOOT']

abt = alvo.copy()
for tabela in [FS_TRANSACOES, FS_PRODUTOS, FS_LIVES, FS_CURSOS]:
    fs = spark.table(tabela).toPandas()
    abt = abt.merge(fs, on=['dtRef', 'IdCliente'], how='left')

faltando = abt[ORDEM[3:-2]].isna().all(axis=1).sum()
assert faltando == 0, f'{faltando} linhas sem features: rode o 02a_feature_store para essas safras'

## 3. Rótulo das safras

In [0]:
dt = pd.to_datetime(abt['dtRef'])
abt['baseSafra'] = np.select(
    [dt < pd.Timestamp(DT_OOT),
     dt == pd.Timestamp(DT_OOT),
     dt == pd.Timestamp(DT_SCORE)],
    ['treino', 'oot', 'score'],
    'pos_oot')                                                  # entre a OOT e o score
abt['flOOT'] = (abt['baseSafra'] == 'oot').astype(int)          # mantido por compatibilidade
abt['churn'] = abt['churn'].astype(float)
abt.loc[abt['baseSafra'] == 'score', 'churn'] = np.nan          # rótulo incompleto: ainda não há 28 dias de futuro

abt = abt[ORDEM]
assert not abt.duplicated(['dtRef', 'IdCliente']).any(), 'ABT com usuário × safra duplicado'
abt.groupby('baseSafra').agg(safras=('dtRef', 'nunique'), linhas=('IdCliente', 'size'), churn=('churn', 'mean')).round(3)

## 4. Conferência com a ABT anterior
Na primeira execução, compara com a ABT montada pelo `02_abt` antigo. Se tudo bater, o modelo do `05`/`06` continua válido e não precisa ser treinado de novo.

In [0]:
if spark.catalog.tableExists(TABELA_ABT):
    chave = ['dtRef', 'IdCliente']
    antiga = spark.table(TABELA_ABT).toPandas().sort_values(chave).reset_index(drop=True)
    nova   = abt.sort_values(chave).reset_index(drop=True)

    print(f'Linhas: antiga {len(antiga):,} | nova {len(nova):,}'.replace(',', '.'))
    print(f'Colunas só na antiga: {sorted(set(antiga.columns) - set(nova.columns))}')
    print(f'Colunas só na nova:   {sorted(set(nova.columns) - set(antiga.columns))}')
    mesmas_linhas = len(antiga) == len(nova) and (antiga[chave].astype(str).values == nova[chave].astype(str).values).all()
    print(f'Mesmos usuários e safras: {mesmas_linhas}')

    if mesmas_linhas:
        difs = {}
        for c in [c for c in nova.columns if c in antiga.columns and c not in chave]:
            a, n = antiga[c], nova[c]
            if pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(n):
                nulo_diferente = (a.isna() != n.isna()).sum()
                valor_diferente = ((a - n).abs() > 1e-9).sum()
                difs[c] = nulo_diferente + valor_diferente
            else:
                difs[c] = (a.astype(str) != n.astype(str)).sum()
        difs = pd.Series(difs, name='linhas diferentes')
        if difs.sum():
            print('Colunas com diferença:')
            display(difs[difs > 0].to_frame())
        else:
            print('Todas as colunas batem ✅')

## 5. Salvar a ABT

In [0]:
spark.createDataFrame(abt).write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable(TABELA_ABT)
print(f'{TABELA_ABT}: {len(abt):,} linhas, {len(abt.columns)} colunas'.replace(',', '.'))